# Logging and Monitoring Deployed ML Models

Deploying a model is not the end of the work — it is the beginning. Once your model is serving real traffic, you need to know:
- Is it responding? (availability)
- How fast is it responding? (latency)
- Are predictions drifting from what the training distribution would suggest? (model health)

This notebook simulates a production logging and monitoring setup locally. All patterns apply directly to CloudWatch (AWS), Cloud Monitoring (GCP), and Azure Monitor.

## Learning Objectives

By the end of this notebook you will be able to:
1. Emit structured JSON logs from a prediction function
2. Simulate a realistic stream of prediction requests with varying latency
3. Compute p50, p95, p99 latency and explain the difference
4. Plot latency over time and a prediction distribution histogram
5. Explain how confidence score distribution can proxy for model accuracy monitoring

## Section 1 — Structured Logging

Plain-text logs are hard to query. Structured logs emit each record as JSON so tools like CloudWatch Insights, BigQuery, or Elasticsearch can filter by field.

Every prediction log record should include:
- **timestamp** — when did the request arrive?
- **request_id** — unique ID to trace one request across services
- **input_features** — what did the caller send? (needed to replay and debug)
- **prediction** and **confidence** — what did the model return?
- **latency_ms** — how long did inference take?
- **model_version** — which model produced this output?

In [ ]:
import logging
import json
import uuid
import time
import pathlib
from datetime import datetime, timezone

LOG_FILE = pathlib.Path('/tmp/ml_predictions.jsonl')
LOG_FILE.unlink(missing_ok=True)  # start fresh

class JSONFormatter(logging.Formatter):
    """Emit each log record as a single JSON line."""
    def format(self, record):
        log_obj = {
            'timestamp': datetime.fromtimestamp(record.created, tz=timezone.utc).isoformat(),
            'level': record.levelname,
            'logger': record.name,
        }
        # Merge any extra fields passed to the logger
        if hasattr(record, 'extra'):
            log_obj.update(record.extra)
        else:
            log_obj['message'] = record.getMessage()
        return json.dumps(log_obj)

# Set up logger
logger = logging.getLogger('ml_api')
logger.setLevel(logging.INFO)
logger.handlers.clear()

file_handler = logging.FileHandler(LOG_FILE)
file_handler.setFormatter(JSONFormatter())
logger.addHandler(file_handler)

console_handler = logging.StreamHandler()
console_handler.setFormatter(JSONFormatter())
logger.addHandler(console_handler)

def log_prediction(features, prediction, confidence, latency_ms, model_version='1.0.0', error=None):
    """Log a prediction event as a structured JSON record."""
    record = logging.LogRecord(
        name='ml_api', level=logging.INFO,
        pathname='', lineno=0, msg='', args=(), exc_info=None
    )
    record.extra = {
        'event': 'prediction',
        'request_id': str(uuid.uuid4())[:8],
        'model_version': model_version,
        'input_features': features,
        'prediction': prediction,
        'confidence': confidence,
        'latency_ms': latency_ms,
        'error': error,
    }
    logger.handle(record)

# Log a single prediction to see what the record looks like
log_prediction(
    features=[5.1, 3.5, 1.4, 0.2],
    prediction='setosa',
    confidence=0.97,
    latency_ms=12.3,
)

## Section 2 — Simulate 500 Prediction Requests

We train a model and run 500 simulated requests through it, logging every result. Latency is simulated with random noise to mimic real network and compute variability.

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Train model
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler().fit(X_train)
clf = RandomForestClassifier(n_estimators=100, random_state=42).fit(scaler.transform(X_train), y_train)

print(f"Model ready. Test accuracy: {clf.score(scaler.transform(X_test), y_test):.2%}")

# Reset logging for the bulk simulation. The file handler from the setup cell
# still holds the original file descriptor, so unlinking the path out from under
# it would send writes to a deleted inode (and LOG_FILE.stat() would then fail).
# Detach the old handlers, start a fresh file, and skip the console handler so we
# don't print 500 JSON lines.
for _h in list(logger.handlers):
    _h.close()
    logger.removeHandler(_h)
LOG_FILE.unlink(missing_ok=True)
_bulk_handler = logging.FileHandler(LOG_FILE)
_bulk_handler.setFormatter(JSONFormatter())
logger.addHandler(_bulk_handler)

N_REQUESTS = 500
rng = np.random.default_rng(seed=42)

# Simulate requests: mostly from the test set, some random noise
for i in range(N_REQUESTS):
    # Sample a random test point (or add slight noise)
    base_features = X_test[i % len(X_test)]
    noisy_features = base_features + rng.normal(0, 0.05, size=base_features.shape)
    noisy_features = np.clip(noisy_features, X.min(axis=0), X.max(axis=0))

    # Simulate inference time: mostly 10-30ms, occasional spikes
    latency = rng.exponential(15) + rng.choice([0, 100], p=[0.97, 0.03])
    latency = round(float(np.clip(latency, 1, 500)), 2)

    start = time.perf_counter()
    features_s = scaler.transform(noisy_features.reshape(1, -1))
    pred = int(clf.predict(features_s)[0])
    conf = float(clf.predict_proba(features_s)[0].max())
    _ = time.perf_counter() - start  # actual inference time

    log_prediction(
        features=noisy_features.round(3).tolist(),
        prediction=iris.target_names[pred],
        confidence=round(conf, 3),
        latency_ms=latency,
        model_version='1.0.0',
    )

print(f"Simulated {N_REQUESTS} requests. Log written to {LOG_FILE}")
print(f"Log file size: {LOG_FILE.stat().st_size / 1024:.1f} KB")

## Section 3 — Parse Logs and Compute Metrics

In [ ]:
import json
from collections import Counter

# Load all log records
records = []
with open(LOG_FILE) as f:
    for line in f:
        line = line.strip()
        if line:
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                pass

# Filter to prediction events only
pred_records = [r for r in records if r.get('event') == 'prediction']
latencies = np.array([r['latency_ms'] for r in pred_records])
predictions = [r['prediction'] for r in pred_records]
confidences = np.array([r['confidence'] for r in pred_records])
error_count = sum(1 for r in pred_records if r.get('error'))

# Latency percentiles
p50 = np.percentile(latencies, 50)
p95 = np.percentile(latencies, 95)
p99 = np.percentile(latencies, 99)

print(f"Total predictions : {len(pred_records)}")
print(f"Error count       : {error_count} ({error_count/len(pred_records)*100:.1f}%)")
print()
print("Latency statistics (ms):")
print(f"  Mean : {latencies.mean():.1f}")
print(f"  p50  : {p50:.1f}  (50% of requests faster than this)")
print(f"  p95  : {p95:.1f}  (95% of requests faster than this)")
print(f"  p99  : {p99:.1f}  (99% of requests faster than this)")
print(f"  Max  : {latencies.max():.1f}")
print()
print("Prediction distribution:")
for label, count in sorted(Counter(predictions).items()):
    pct = count / len(predictions) * 100
    print(f"  {label:12s}: {count:4d} ({pct:.1f}%)")
print()
print(f"Mean confidence   : {confidences.mean():.3f}")
print(f"Min confidence    : {confidences.min():.3f}")

## Section 4 — Visualize Latency and Prediction Distribution

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: Latency over time (rolling average)
ax = axes[0]
window = 20
rolling_avg = np.convolve(latencies, np.ones(window)/window, mode='valid')
ax.plot(latencies, alpha=0.3, color='steelblue', linewidth=0.5, label='Raw')
ax.plot(range(window-1, len(latencies)), rolling_avg, color='navy', linewidth=1.5, label=f'{window}-req avg')
ax.axhline(p95, color='orange', linestyle='--', linewidth=1, label=f'p95={p95:.0f}ms')
ax.axhline(p99, color='red', linestyle='--', linewidth=1, label=f'p99={p99:.0f}ms')
ax.set_title('Latency Over Time')
ax.set_xlabel('Request number')
ax.set_ylabel('Latency (ms)')
ax.legend(fontsize=8)
ax.set_ylim(0, min(latencies.max() * 1.1, 300))

# Plot 2: Latency distribution
ax = axes[1]
ax.hist(latencies, bins=40, color='steelblue', edgecolor='white', linewidth=0.5)
ax.axvline(p50, color='green', linestyle='--', label=f'p50={p50:.0f}ms')
ax.axvline(p95, color='orange', linestyle='--', label=f'p95={p95:.0f}ms')
ax.axvline(p99, color='red', linestyle='--', label=f'p99={p99:.0f}ms')
ax.set_title('Latency Distribution')
ax.set_xlabel('Latency (ms)')
ax.set_ylabel('Count')
ax.legend(fontsize=8)

# Plot 3: Prediction distribution
ax = axes[2]
pred_counts = Counter(predictions)
labels = sorted(pred_counts.keys())
counts = [pred_counts[l] for l in labels]
colors = ['#4C72B0', '#DD8452', '#55A868']
bars = ax.bar(labels, counts, color=colors[:len(labels)], edgecolor='white', linewidth=0.5)
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
            str(count), ha='center', va='bottom', fontsize=9)
ax.set_title('Prediction Distribution')
ax.set_ylabel('Count')

plt.suptitle(f'ML API Monitoring Dashboard — {len(pred_records)} requests', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/monitoring_dashboard.png', dpi=100, bbox_inches='tight')
plt.show()
print('Dashboard saved to /tmp/monitoring_dashboard.png')

## Section 5 — Cloud Monitoring Overview

The patterns above run locally, but cloud platforms offer managed equivalents:

**AWS CloudWatch**
- Your application emits JSON logs to CloudWatch Logs
- CloudWatch Insights lets you query logs with SQL-like syntax:
  ```
  fields @timestamp, latency_ms, prediction
  | filter event = 'prediction'
  | stats pct(latency_ms, 99) as p99 by bin(5m)
  ```
- CloudWatch Alarms trigger when a metric exceeds a threshold (e.g., p99 > 500ms for 5 minutes)

**GCP Cloud Monitoring + Cloud Logging**
- Structured logs flow to Cloud Logging automatically from Cloud Run / GKE
- Log-based metrics convert log field values into Monitoring time series
- Equivalent query in Cloud Logging:
  ```
  resource.type="cloud_run_revision"
  jsonPayload.event="prediction"
  jsonPayload.latency_ms > 200
  ```

**Azure Monitor + Log Analytics**
- Logs go to a Log Analytics workspace
- Query with KQL (Kusto Query Language):
  ```kql
  customEvents
  | where name == "prediction"
  | summarize p99=percentile(todouble(customDimensions.latency_ms), 99) by bin(timestamp, 5m)
  ```

The JSON structure you log locally is the same structure these platforms index and query.

## Section 6 — Model Accuracy Monitoring Without Ground Truth

The hard problem: after deployment, you receive inputs but often do not have ground-truth labels to evaluate accuracy. You cannot compute accuracy directly.

**Proxy signals that indicate model degradation:**
1. **Confidence score distribution** — if the model's confidence drops, it is seeing unfamiliar inputs
2. **Prediction distribution shift** — if 90% of predictions suddenly become class A, something changed
3. **Input feature distribution** — if input values drift outside the training range, the model extrapolates

In [ ]:
import numpy as np
from scipy import stats

# Simulate a "healthy" period and a "degraded" period
rng = np.random.default_rng(42)

# Healthy: confidence scores from real iris-like distribution
healthy_confidences = rng.beta(9, 1, size=200)  # high confidence, centered near 0.9

# Degraded: confidence drops as model sees out-of-distribution inputs
degraded_confidences = rng.beta(3, 3, size=200)  # much more uncertain

# Kolmogorov-Smirnov test: detects distributional shift
ks_stat, ks_pvalue = stats.ks_2samp(healthy_confidences, degraded_confidences)

print("Confidence Score Distribution Comparison:")
print(f"  Healthy period   — mean={healthy_confidences.mean():.3f}, std={healthy_confidences.std():.3f}")
print(f"  Degraded period  — mean={degraded_confidences.mean():.3f}, std={degraded_confidences.std():.3f}")
print()
print("Kolmogorov-Smirnov drift test:")
print(f"  KS statistic : {ks_stat:.3f}  (0 = identical, 1 = completely different)")
print(f"  p-value      : {ks_pvalue:.2e}")
print(f"  Drift detected: {'YES — investigate the model' if ks_pvalue < 0.05 else 'No significant drift'}")
print()
print("In production: run this test every hour/day comparing the last N predictions")
print("to a reference window from a known-good period.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confidence distribution comparison
ax = axes[0]
ax.hist(healthy_confidences, bins=30, alpha=0.6, color='green', label='Healthy period', density=True)
ax.hist(degraded_confidences, bins=30, alpha=0.6, color='red', label='Degraded period', density=True)
ax.axvline(healthy_confidences.mean(), color='darkgreen', linestyle='--', linewidth=1.5)
ax.axvline(degraded_confidences.mean(), color='darkred', linestyle='--', linewidth=1.5)
ax.set_title('Confidence Score Distribution Shift')
ax.set_xlabel('Confidence')
ax.set_ylabel('Density')
ax.legend()

# Rolling mean confidence over time (simulated)
ax = axes[1]
all_conf = np.concatenate([healthy_confidences, degraded_confidences])
rolling_conf = np.convolve(all_conf, np.ones(20)/20, mode='valid')
t = range(len(rolling_conf))
ax.plot(t, rolling_conf, color='steelblue', linewidth=1.5)
ax.axvline(len(healthy_confidences) - 10, color='red', linestyle='--', label='Drift starts here')
ax.axhline(0.8, color='orange', linestyle=':', label='Alert threshold')
ax.set_title('Rolling Mean Confidence (Drift Detection)')
ax.set_xlabel('Request number')
ax.set_ylabel('Mean confidence (20-req window)')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('/tmp/drift_detection.png', dpi=100, bbox_inches='tight')
plt.show()
print('Drift detection chart saved to /tmp/drift_detection.png')

## Summary

**What to log** for every prediction:
- Timestamp, request ID, model version
- Input features (for replay and debugging)
- Prediction and confidence score
- Latency in milliseconds
- Any error message

**Why log input features**: without them you cannot replay a failed request, debug an unexpected prediction, or run distribution shift analysis on the inputs themselves.

**Latency percentiles**:
- p50 = median — most users experience this
- p95 = 95th percentile — 1 in 20 users experiences this or worse
- p99 = 99th percentile — the worst 1% of requests; what your worst-case user sees

**Monitoring without ground truth**: use confidence score distributions and prediction class distributions as proxy signals. A sudden drop in mean confidence or shift in class distribution indicates the model is seeing unfamiliar data and should be investigated.

## Self-Check

1. **What is the difference between p50 and p99 latency?**
   *(Which one matters more for user experience? When would you care about p99?)*

2. **Why is logging the input features (not just the predictions) important?**
   *(Think about what you can do with the logged features that you cannot do with just the prediction.)*

3. **If you don't have ground-truth labels for new data, how can you detect model degradation?**
   *(Name at least two proxy signals shown in this notebook.)*